# 03 · Defect detection — RT-DETR-X

Trains and evaluates RT-DETR-X (Ultralytics) on the same four defect classes. Includes micro-crack oversampling, cleanup of orphan labels/caches, and a *recovery* run that restarts from a stable checkpoint with AMP disabled after a NaN divergence.

| | |
|---|---|
| **Input** | YOLO-format dataset (processed, borders removed) + `data.yaml` |
| **Output** | Training curves, validation metrics (mAP@50, precision, recall), exported model |
| **Runs on** | GPU (NVIDIA A100 used for the reported run) |

> **Notes**
> - The recovery cell loads a checkpoint from an earlier, diverged run (`rtdetr_A100_final_v1-2`). To start from scratch, use `RTDETR(MODEL_SIZE)` instead.
> - The results and predict cells use the latest `rtdetr_*` run folder, and the prediction source is the `valid/` split.
> - Weights and datasets are not included in this repository (see `DATA.md`).

# RT-DETR Training on YOLO11 Format Data

This notebook trains RT-DETR model using YOLO11 format dataset from Google Drive.

**Requirements:**
- Dataset in YOLO11 format (images + labels)
- Dataset stored in Google Drive
- data.yaml configuration file

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
# Install Ultralytics (includes RT-DETR support)
!pip install ultralytics -q

# Verify installation
import ultralytics
ultralytics.checks()

print(f"\nUltralytics version: {ultralytics.__version__}")

## 3. Configuration

Update these paths to match your Google Drive structure:

In [ ]:
# ========== CONFIGURE THESE PATHS ==========

# Path to your dataset folder in Google Drive
DATASET_PATH = '/content/drive/MyDrive/inspectra/detection_dataset_processed'

# Path to data.yaml file (should be in your dataset folder)
DATA_YAML ='/content/drive/MyDrive/inspectra/detection_dataset_processed/data.yaml'

# Where to save training results
OUTPUT_PATH = '/content/drive/MyDrive/inspectra/runs/rtdetr'

# RT-DETR model size: rtdetr-l, rtdetr-x
MODEL_SIZE = 'rtdetr-x.pt'

# Training parameters
EPOCHS = 120
IMG_SIZE = 1280
BATCH_SIZE = 4  # Adjust based on GPU memory - Reduced from 16

# ==========================================

## 4. Verify Dataset Structure

In [ ]:
import os
import yaml

# Check if dataset path exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ Error: Dataset path not found: {DATASET_PATH}")
    print("Please update DATASET_PATH in the configuration cell above.")
else:
    print(f"✓ Dataset folder found: {DATASET_PATH}")

# Check data.yaml
if not os.path.exists(DATA_YAML):
    print(f"\n❌ Error: data.yaml not found: {DATA_YAML}")
else:
    print(f"\n✓ data.yaml found: {DATA_YAML}")

    # Display data.yaml contents
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)

    print("\n📋 Dataset Configuration:")
    print(f"  Classes: {data_config.get('nc', 'Not specified')}")
    print(f"  Names: {data_config.get('names', 'Not specified')}")
    print(f"  Train: {data_config.get('train', 'Not specified')}")
    print(f"  Val: {data_config.get('val', 'Not specified')}")

# List dataset structure
print("\n📂 Dataset Structure:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Only show first 2 levels to avoid clutter
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")

## 5. Check GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")

## 6. Initialize RT-DETR Model

In [ ]:
from ultralytics import RTDETR

# Load pretrained RT-DETR model
print(f"Loading {MODEL_SIZE}...")
model = RTDETR(MODEL_SIZE)

print("\n✓ Model loaded successfully!")
print(f"Model: {model.model_name}")

In [ ]:
import os
import shutil

# Paths (taken from the configuration above)
train_labels_path = '/content/drive/MyDrive/inspectra/detection_dataset_processed/train/labels'
train_images_path = '/content/drive/MyDrive/inspectra/detection_dataset_processed/train/images'
minority_class_id = '3' # Micro-crack

def smart_oversample(labels_path, images_path, class_id, factor=2):
    # Only use original files (not starting with aug_) to avoid repeated oversampling
    files = [f for f in os.listdir(labels_path) if f.endswith('.txt') and not f.startswith('aug_')]
    count = 0

    for label_file in files:
        with open(os.path.join(labels_path, label_file), 'r') as f:
            lines = f.readlines()
            if any(line.split()[0] == class_id for line in lines):
                for i in range(1, factor):
                    new_label_name = f"aug_{i}_{label_file}"
                    new_img_name = f"aug_{i}_{label_file.replace('.txt', '.jpg')}"

                    # Copy the files if they do not already exist
                    if not os.path.exists(os.path.join(labels_path, new_label_name)):
                        shutil.copy(os.path.join(labels_path, label_file), os.path.join(labels_path, new_label_name))

                        img_src = os.path.join(images_path, label_file.replace('.txt', '.jpg'))
                        if os.path.exists(img_src):
                            shutil.copy(img_src, os.path.join(images_path, new_img_name))
                        count += 1
    print(f"Added {count} new samples for class {class_id}. The training folder is ready.")

# Run the oversampling
smart_oversample(train_labels_path, train_images_path, minority_class_id, factor=2)

In [ ]:
import os

# Path to the training images folder
train_images_path = f"{DATASET_PATH}/train/images"

all_files = os.listdir(train_images_path)
original_files = [f for f in all_files if not f.startswith('aug_')]
oversampled_files = [f for f in all_files if f.startswith('aug_')]

print(f"Folder statistics:")
print(f"Original images: {len(original_files)}")
print(f"Oversampled images: {len(oversampled_files)}")
print(f"Total: {len(all_files)}")

In [ ]:
import os

# Paths
labels_dir = f"{DATASET_PATH}/train/labels"
images_dir = f"{DATASET_PATH}/train/images"

# Remove any label file that has no matching image
label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
deleted_count = 0

for lf in label_files:
    img_f = lf.replace('.txt', '.jpg')
    if not os.path.exists(os.path.join(images_dir, img_f)):
        os.remove(os.path.join(labels_dir, lf))
        deleted_count += 1

# Delete stale label caches (required so Ultralytics rebuilds them)
for f in os.listdir(labels_dir):
    if f.endswith('.cache'):
        os.remove(os.path.join(labels_dir, f))

print(f"Cleaned {deleted_count} orphan label files and removed the old caches.")

## 7. Train the Model

This will start the training process. Training progress will be displayed below.

In [ ]:
from ultralytics import RTDETR

# 1. Load the best checkpoint saved before the run diverged
# (expected to be the healthy checkpoint from epoch 8)
model = RTDETR('/content/drive/MyDrive/inspectra/runs/rtdetr/rtdetr_A100_final_v1-2/weights/best.pt')

# 2. Start a fresh run from those weights (instead of resuming) to avoid the old NaN values
results = model.train(
    data=DATA_YAML,
    epochs=120,
    imgsz=1280,
    batch=8,
    cache=True,

    # Stability settings to avoid repeating the NaN problem
    lr0=0.0001,      # very low learning rate for stability
    amp=False,       # disable AMP to prevent numerical blow-up
    warmup_epochs=0, # no warmup needed: starting from partially trained weights

    project=OUTPUT_PATH,
    name='rtdetr_A100_recovery_v1',
    device=0
)

In [ ]:
# from requests import patch
# # Create output directory
# os.makedirs(OUTPUT_PATH, exist_ok=True)

# # Start training
# results = model.train(
#  model=MODEL_SIZE,  # Explicitly pass the model size to ensure its definition is recognized
#     data=DATA_YAML,
#     epochs=EPOCHS,
#     imgsz=IMG_SIZE,
#     batch=BATCH_SIZE,
#     project=OUTPUT_PATH,
#     name='yolo11_training',
#     device=0 if torch.cuda.is_available() else 'cpu',
#     # cls=1.2, # raised to focus on classification accuracy [to compensate for scarce samples]
#     augment=True,
#      # enable data augmentation to increase diversity

#     # # Additional training arguments
#     # patience=30,  # Early stopping patience
#     # save=True,    # Save checkpoints
#     # save_period=10,  # Save checkpoint every 10 epochs

#     # # Data augmentation
#     # hsv_h=0.015,  # HSV-Hue augmentation
#     # hsv_s=0.7,    # HSV-Saturation augmentation

#     # #hsv_v=0.4,    # HSV-Value augmentation
#     # hsv_v=0.6,

#     # degrees=0.0,  # Rotation augmentation

#     # translate=0.1,  # Translation augmentation
#     # scale=0.2,    # Scale augmentation


#     # #shear=0.0,    # Shear augmentation
#     # #perspective=0.0,  # Perspective augmentation
#     # shear=2.0,
#     # perspective=0.0005,

#     # #flipud=0.0,   # Vertical flip probability
#     # flipud=0.5,

#     # fliplr=0.5,   # Horizontal flip probability

#     # mosaic=0.2,   # 0.5 Mosaic augmentation probability
#     # mixup=0.0,    # Mixup augmentation probability


#     # copy_paste=0.0,  # Copy-paste augmentation probability

#     # # Optimizer
#     # optimizer='auto',  # auto, SGD, Adam, AdamW, NAdam, RAdam, RMSProp
#     # lr0=0.01,     # Initial learning rate
#     # lrf=0.01,     # Final learning rate factor
#     # momentum=0.937,
#     # weight_decay=0.0005,
#     # warmup_epochs=3.0,
#     # warmup_momentum=0.8,
#     # warmup_bias_lr=0.1,

#     # # Loss weights
#     # box=7.5,      # Box loss weight
#     # # cls=0.5,      # Classification loss weight  <-- This line is removed
#     # dfl=1.5,      # DFL loss weight

#     # # Other
#     # verbose=True,
#     # seed=0,
#     # deterministic=False,
#     # plots=True,   # Save training plots
#     # amp=True,     # Automatic Mixed Precision
#     # fraction=1.0, # Use 100% of dataset

#     # #close_mosaic=10,  # Disable mosaic augmentation in last N epochs
#     # close_mosaic=10, #15,

# )

# print("\n✓ Training completed!")

## 8. Display Training Results

In [ ]:
from IPython.display import Image, display
import glob

# Find the latest training folder
training_folders = glob.glob(f"{OUTPUT_PATH}/rtdetr_*")
if training_folders:
    latest_folder = max(training_folders, key=os.path.getctime)

    print(f"📊 Training results saved to: {latest_folder}\n")

    # Display training curves
    results_img = f"{latest_folder}/results.png"
    if os.path.exists(results_img):
        print("Training Curves:")
        display(Image(filename=results_img, width=1000))

    # Display confusion matrix
    confusion_matrix = f"{latest_folder}/confusion_matrix.png"
    if os.path.exists(confusion_matrix):
        print("\nConfusion Matrix:")
        display(Image(filename=confusion_matrix, width=800))

    # Display sample predictions
    val_batch = f"{latest_folder}/val_batch0_pred.jpg"
    if os.path.exists(val_batch):
        print("\nSample Validation Predictions:")
        display(Image(filename=val_batch, width=1000))
else:
    print("No training results found.")

## 9. Validate the Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print("\n📈 Validation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
# print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 10. Export the Model

In [ ]:
# Export to different formats
export_folder = f"{OUTPUT_PATH}/exported_models"
os.makedirs(export_folder, exist_ok=True)

print("Exporting model...\n")

# Export to ONNX (recommended for deployment)
try:
    onnx_path = model.export(format='onnx', dynamic=True)
    print(f"✓ ONNX model exported: {onnx_path}")
except Exception as e:
    print(f"❌ ONNX export failed: {e}")

# Export to TorchScript
try:
    torchscript_path = model.export(format='torchscript')
    print(f"✓ TorchScript model exported: {torchscript_path}")
except Exception as e:
    print(f"❌ TorchScript export failed: {e}")

print(f"\nBest weights saved at: {latest_folder}/weights/best.pt")

## 11. Test Inference on Sample Images

In [ ]:
# Load the best model
best_model = RTDETR(f"{latest_folder}/weights/best.pt")

# Run inference on validation images
test_results = best_model.predict(
    source=f"{DATASET_PATH}/valid/images",  # Adjust path as needed
    conf=0.25,  # Confidence threshold
    save=True,
    save_txt=True,
    save_conf=True,
    project=f"{OUTPUT_PATH}/predictions",
    name='test_predictions',
    max_det=300,  # Maximum detections per image
)

print("\n✓ Predictions saved!")

# Display sample predictions
pred_images = glob.glob(f"{OUTPUT_PATH}/predictions/test_predictions/*.jpg")
if pred_images:
    print("\nSample Predictions:")
    for img_path in pred_images[:3]:  # Show first 3 predictions
        display(Image(filename=img_path, width=800))
        print()
else:
    print("No prediction images found.")

## 12. Summary & Next Steps

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\n✓ Model: RT-DETR {MODEL_SIZE}")
print(f"✓ Epochs completed: {EPOCHS}")
print(f"✓ Final mAP50-95: {metrics.box.map:.4f}")
print(f"✓ Final mAP50: {metrics.box.map50:.4f}")
print(f"\n📁 Results saved to: {OUTPUT_PATH}")
print(f"📁 Best weights: {latest_folder}/weights/best.pt")
print(f"📁 Last weights: {latest_folder}/weights/last.pt")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("1. Review training curves and metrics above")
print("2. Test on your own images using the inference section")
print("3. Export model for deployment (ONNX, TorchScript, etc.)")
print("4. Download weights from Google Drive for local use")
print("5. Fine-tune hyperparameters if needed and retrain")
print("\n" + "="*60)